[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/01-os-filesystem.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/01-os-filesystem.ipynb)

# Module 3.1 — OS & Filesystem Operations
**Module 3: Automation & Scripting** | Estimated time: 30 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Use both `os` / `os.path` and the modern `pathlib.Path` API
- Perform common file and directory operations (create, list, rename, delete)
- Copy and move files with `shutil`
- Walk directory trees with `os.walk()`
- Use `glob` patterns to find files by name or extension
- Build a directory organizer script that sorts files by extension

In [ ]:
# Setup — all modules are part of the standard library, no pip install needed
import os
import os.path
import shutil
import glob
from pathlib import Path
import tempfile

# We will work inside /tmp/pypath_demo so nothing important is touched
BASE = Path('/tmp/pypath_demo')
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir(parents=True)
print('Working directory:', BASE)

## 1. `os.path` vs `pathlib.Path` — Side-by-Side Comparison

`os.path` treats paths as plain strings and uses function calls.  
`pathlib.Path` wraps paths in an object so you can chain methods and use the `/` operator to join segments.  Both are fine; `pathlib` is preferred in modern Python (3.6+).

In [ ]:
# ── os.path style ──────────────────────────────────────────────────────────
path_str = '/tmp/pypath_demo/reports/summary.txt'
print('dirname :', os.path.dirname(path_str))
print('basename:', os.path.basename(path_str))
print('splitext:', os.path.splitext(path_str))
print('join    :', os.path.join('/tmp', 'pypath_demo', 'data', 'file.csv'))
print('exists  :', os.path.exists(path_str))
print()

# ── pathlib.Path style ─────────────────────────────────────────────────────
p = Path('/tmp/pypath_demo/reports/summary.txt')
print('parent  :', p.parent)
print('name    :', p.name)
print('stem    :', p.stem)
print('suffix  :', p.suffix)
print('parts   :', p.parts)
print('join (/):', Path('/tmp') / 'pypath_demo' / 'data' / 'file.csv')
print('exists  :', p.exists())

## 2. Creating Directories and Files with `pathlib`

Key `Path` methods:
| Method | Description |
|---|---|
| `mkdir(parents=True, exist_ok=True)` | Create directory (and parents) |
| `touch()` | Create an empty file |
| `write_text(s)` | Write a string to a file |
| `read_text()` | Read a file as a string |
| `write_bytes(b)` | Write raw bytes |
| `read_bytes()` | Read raw bytes |
| `unlink()` | Delete a file |
| `rmdir()` | Delete an empty directory |

In [ ]:
# Create a demo directory structure
for folder in ['docs', 'images', 'scripts', 'data/raw', 'data/processed']:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

# Create sample files
files = {
    'docs/readme.txt'      : 'Project documentation.',
    'docs/notes.txt'       : 'Meeting notes for Q1.',
    'docs/report.pdf'      : '%PDF-1.4 fake pdf content',
    'images/logo.png'      : 'PNG fake binary',
    'images/banner.jpg'    : 'JFIF fake binary',
    'images/icon.png'      : 'PNG fake binary 2',
    'scripts/build.py'     : 'print("build")',
    'scripts/deploy.sh'    : '#!/bin/bash\necho deploy',
    'data/raw/sales.csv'   : 'date,amount\n2024-01-01,100',
    'data/raw/users.json'  : '{"users": []}',
}
for rel, content in files.items():
    p = BASE / rel
    p.write_text(content)

print('Created files:')
for p in sorted(BASE.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(BASE))

## 3. Inspecting Paths — `is_file`, `is_dir`, `stat`, `iterdir`

In [ ]:
p = BASE / 'docs'
print('is_dir  :', p.is_dir())
print('is_file :', p.is_file())
print()

# List immediate children
print('Contents of docs/:')
for child in sorted(p.iterdir()):
    stat = child.stat()
    kind = 'DIR ' if child.is_dir() else 'FILE'
    print(f'  [{kind}] {child.name:30s}  {stat.st_size:6d} bytes')
print()

# Resolve absolute path and check symlink status
script = BASE / 'scripts' / 'build.py'
print('absolute:', script.resolve())
print('is_symlink:', script.is_symlink())

## 4. Glob Patterns — Finding Files by Pattern

`Path.glob(pattern)` matches files in the current directory.  
`Path.rglob(pattern)` is recursive (equivalent to `**/pattern`).  
You can also use the `glob` module directly.

In [ ]:
# Find all .txt files anywhere under BASE
print('All .txt files (rglob):')
for f in sorted(BASE.rglob('*.txt')):
    print(' ', f.relative_to(BASE))
print()

# Find only Python files
print('Python files:')
for f in BASE.rglob('*.py'):
    print(' ', f.relative_to(BASE))
print()

# glob module — useful when you have a string pattern
pattern = str(BASE / 'images' / '*.png')
print('PNG files via glob.glob:')
for f in glob.glob(pattern):
    print(' ', Path(f).relative_to(BASE))
print()

# Wildcard in directory name
print('All files in data/**:')
for f in glob.glob(str(BASE / 'data' / '**' / '*'), recursive=True):
    if os.path.isfile(f):
        print(' ', Path(f).relative_to(BASE))

## 5. Walking Directory Trees with `os.walk()`

`os.walk(top)` yields `(dirpath, dirnames, filenames)` for every directory in the tree. It is the classic approach and is still useful when you need fine-grained control over recursion.

In [ ]:
print(f'Tree rooted at {BASE}:\n')
for dirpath, dirnames, filenames in os.walk(BASE):
    # depth for indentation
    depth = dirpath.replace(str(BASE), '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(dirpath)}/')
    sub_indent = '  ' * (depth + 1)
    for fname in filenames:
        full = os.path.join(dirpath, fname)
        size = os.path.getsize(full)
        print(f'{sub_indent}{fname}  ({size} B)')

## 6. `shutil` — Copy, Move, and Delete

| Function | Description |
|---|---|
| `shutil.copy(src, dst)` | Copy file (no metadata) |
| `shutil.copy2(src, dst)` | Copy file + metadata |
| `shutil.move(src, dst)` | Move / rename |
| `shutil.copytree(src, dst)` | Copy entire directory tree |
| `shutil.rmtree(path)` | Delete directory tree |
| `shutil.make_archive(base, fmt, root)` | Create zip/tar archive |

In [ ]:
# Copy a file
shutil.copy(BASE / 'docs' / 'readme.txt', BASE / 'docs' / 'readme_backup.txt')
print('Copied readme.txt -> readme_backup.txt')

# Move (rename) a file
shutil.move(str(BASE / 'docs' / 'notes.txt'), str(BASE / 'docs' / 'notes_archived.txt'))
print('Moved notes.txt -> notes_archived.txt')

# Copy an entire directory tree
shutil.copytree(BASE / 'images', BASE / 'images_backup')
print('Copied images/ -> images_backup/')

# Verify
print('\ndocs/ contents now:')
for f in sorted((BASE / 'docs').iterdir()):
    print(' ', f.name)

# Create a zip archive of the docs folder
archive = shutil.make_archive(
    base_name=str(BASE / 'docs_archive'),
    format='zip',
    root_dir=BASE,
    base_dir='docs'
)
print('\nArchive created:', Path(archive).name, f'({Path(archive).stat().st_size} bytes)')

# Remove the backup directory
shutil.rmtree(BASE / 'images_backup')
print('Removed images_backup/')

## 7. Practical Project — Directory Organizer Script

This script scans a target folder and moves files into sub-folders named after their extension.  It is a classic automation task you can run as a cron job or a one-off cleanup utility.

In [ ]:
# First, seed a messy 'inbox' folder with random files
inbox = BASE / 'inbox'
inbox.mkdir(exist_ok=True)

messy_files = [
    'vacation.jpg', 'resume.pdf', 'budget.xlsx', 'script.py',
    'notes.txt', 'logo.png', 'data.csv', 'config.yaml',
    'slideshow.pptx', 'archive.zip', 'main.py', 'report.pdf',
]
for fname in messy_files:
    (inbox / fname).write_text(f'contents of {fname}')

print('Before organizing:')
for f in sorted(inbox.iterdir()):
    print(' ', f.name)

In [ ]:
def organize_directory(source: Path, dest: Path | None = None) -> dict:
    """
    Move every file in *source* into a sub-folder named after its extension.
    If *dest* is None the sub-folders are created inside *source*.
    Returns a summary dict {extension: [filenames]}.
    """
    if dest is None:
        dest = source
    dest.mkdir(parents=True, exist_ok=True)

    summary: dict[str, list[str]] = {}

    for item in sorted(source.iterdir()):
        if not item.is_file():
            continue  # skip sub-directories

        ext = item.suffix.lstrip('.').lower() or 'no_extension'
        target_dir = dest / ext
        target_dir.mkdir(exist_ok=True)

        target_path = target_dir / item.name
        # Avoid overwriting: append a counter if the name already exists
        counter = 1
        while target_path.exists():
            target_path = target_dir / f'{item.stem}_{counter}{item.suffix}'
            counter += 1

        shutil.move(str(item), str(target_path))
        summary.setdefault(ext, []).append(item.name)

    return summary


outbox = BASE / 'organized'
summary = organize_directory(inbox, outbox)

print('Organizer results:')
for ext, files in sorted(summary.items()):
    print(f'  .{ext:10s} -> {len(files)} file(s): {files}')

print('\nOrganized tree:')
for p in sorted(outbox.rglob('*')):
    depth = len(p.relative_to(outbox).parts)
    print('  ' * depth + p.name + ('/' if p.is_dir() else ''))

## Practice Exercises

**Exercise 1 — Rename Files**  
In `/tmp/pypath_demo/organized/txt/`, rename every `.txt` file so that its name is prefixed with `archive_` (e.g. `notes.txt` becomes `archive_notes.txt`). Use `pathlib` only (no `os.rename`).

**Exercise 2 — Disk Usage Report**  
Write a function `disk_usage(path: Path) -> dict` that walks the directory tree at `path` and returns a dictionary mapping each file extension (e.g. `'.py'`) to the total number of bytes used by files with that extension. Print the results sorted from largest to smallest.

**Exercise 3 — Selective Backup**  
Write a function `backup_recent(src: Path, dst: Path, days: int = 7)` that copies only files modified within the last `days` days from `src` to `dst`, preserving the subdirectory structure. Use `shutil.copy2` to preserve timestamps. Hint: use `Path.stat().st_mtime` and `time.time()`.